# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://google.com)


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

The question: *which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?* I'm picking this lane over the others for three reasons. First, it's the lane the starter playground already proves end-to-end — the pipeline in this repo (`scripts/01`–`05`) builds a transparent baseline, trains a model, and exports a ranked queue, so I can see on real numbers (below) that a learned score beats a fixed rule before I commit seven weeks to it. Second, it produces the artifact a content team actually wants: not a report about correlations, but a ranked list someone can work down. Third, it sits in the sweet spot of the density table in the lane guide — search-performance signals (impressions, clicks, position) are the deepest-covered metrics in the warehouse, so a scoring/ranking lane built on them won't starve for data the way the AI-referral freestyle direction would (30,177 AI-session rows vs. 28.9M impression rows).


In [1]:
# A short record of the provisional pick, so later notebooks can reference it programmatically.
lane_choice = {
    "lane": "Lane 2 -- Refresh / Content Opportunity Scoring",
    "status": "provisional, revisit by end of Week 4",
    "core_question": "Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?",
}
lane_choice


{'lane': 'Lane 2 -- Refresh / Content Opportunity Scoring',
 'status': 'provisional, revisit by end of Week 4',
 'core_question': 'Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?'}

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one content item (page) — the same grain as one row in the starter CSV and in `dim_content` in the warehouse. Not a client, not a query, not a day: a page.

**Decision improved:** which page a content reviewer opens first. FlyRank's clients have far more indexed content than review hours. The decision isn't "is this page bad" (binary, and mostly wrong) — it's "out of everything visible right now, which handful is worth a human's next hour."

**Who acts, and how:** a content/SEO reviewer (or the account manager who queues work for one) works down a ranked list, opens the top items, reads the reason codes attached to each (e.g. `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`), and decides per-page whether to refresh, expand, protect, prune, or just keep monitoring. The output is the ranked queue itself, not a single automated action — a human stays in the loop for every page.

**Output:** a ranked review queue — page id, a priority score, an action suggestion (refresh / expand / protect / prune / monitor), and reason codes a reviewer can sanity-check before acting.

**Cost of a wrong call, in both directions:**
- *False positive* (queue says "review this," page was actually fine): a reviewer spends 15–30 minutes opening and checking a page that didn't need it. Annoying, not dangerous — but it burns the exact scarce resource (reviewer time) the whole system exists to protect, and it erodes trust in the queue if it happens often.
- *False negative* (a genuinely declining, high-demand page never surfaces near the top): the page keeps losing visibility/clicks silently until someone notices by accident, weeks or months later. That's real, compounding traffic loss on a page that mattered.
- Because reviewer time is the scarce resource and the queue is capacity-limited (a reviewer works through the top 20–50, not all 30,000 rows), **precision in the top of the list matters more than overall accuracy** — which is exactly why the lane guide points at precision@K instead of a generic accuracy score.




In [2]:
# aligned with the framing-ml-problems skill
import pandas as pd

frame = pd.DataFrame([
    {"question": "Decision", "answer": "Which page should be reviewed first?"},
    {"question": "Actor", "answer": "Content/SEO reviewer or account manager"},
    {"question": "Action", "answer": "Refresh, expand, protect, prune, or monitor"},
    {"question": "Task type", "answer": "Ranking / scoring"},
    {"question": "Output", "answer": "Priority score for review order"},
    {"question": "Primary cost", "answer": "Missed high-impact page or wasted review time"},
    {"question": "Claim level", "answer": "Decision-support, not causal"},
])

frame


,question,answer
0,Decision,Which page should be reviewed first?
1,Actor,Content/SEO reviewer or account manager
2,Action,"Refresh, expand, protect, prune, or monitor"
3,Task type,Ranking / scoring
4,Output,Priority score for review order
5,Primary cost,Missed high-impact page or wasted review time
6,Claim level,"Decision-support, not causal"


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers, computed directly below:

1. **The decline label isn't rare or trivial.** `is_declining_label` (`trend_direction == "down"`) is true for a bit over half of the 30,000 rows -- a real, sizeable pattern to rank against, not a needle-in-a-haystack problem.
2. **A simple rule alone already finds thousands of plausible review candidates**, just from `stale_visible_page` (stale *and* still getting real search visibility) and `declining_with_demand` (falling *and* still has demand) -- more candidates than any reviewer could work through unranked, which is exactly the triage problem this lane exists to solve.
3. **The starter pipeline in this repo already shows a learned score beats the fixed rule**, on the metric that matters for a capacity-limited queue: precision in the top 50. I read this straight from the repo's own verified output (`outputs/model_report.md`) rather than re-deriving it, since it was already run end-to-end.


In [8]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/main/data/raw/content_refresh_anonymized.csv")

print(f"rows: {len(df):,}")
print(f"columns: {df.shape[1]}")
print(f"distinct clients: {df['client_id'].nunique()}")

# Candidate review pages: traffic + stale content
traffic_and_stale = (
    (df["impressions_90d"] >= 1000) &
    (df["days_since_last_update"] >= 180)
)

# High impressions but weak CTR (remember: CTR is stored as percentage points)
high_impr_low_ctr = (
    (df["impressions_90d"] >= 5000) &
    (df["ctr"] < 1.0)   # 1.0 means 1%
)

print("\n1) Pages with >=1,000 impressions and >=180 days since update")
print(f"   {traffic_and_stale.sum():,} pages ({traffic_and_stale.mean():.1%})")

print("\n2) Pages with >=5,000 impressions and CTR < 1%")
print(f"   {high_impr_low_ctr.sum():,} pages ({high_impr_low_ctr.mean():.1%})")

print("\n3) Median days since update")
print(f"   {df['days_since_last_update'].median():.0f} days")

rows: 30,000
columns: 44
distinct clients: 32

1) Pages with >=1,000 impressions and >=180 days since update
   12 pages (0.0%)

2) Pages with >=5,000 impressions and CTR < 1%
   5,901 pages (19.7%)

3) Median days since update
   20 days


In [13]:
# avg_position == 0 means "no data", not rank 0
zero_position = (df["avg_position"] == 0).sum()

# Missingness patterns
missing_word_count = df["word_count"].isna().sum()

print(f"avg_position == 0 (no data): {zero_position:,} rows")
print(f"Missing word_count: {missing_word_count:,} rows")


avg_position == 0 (no data): 1,205 rows
Missing word_count: 7,699 rows


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim, if the results hold up through validation:**
- *Observed:* these pages, in this 90-day window, show these measurable patterns (falling impressions/clicks, stale updates alongside real traffic, weak CTR for their position tier).
- *Decision-support:* a page ranked near the top of the queue is, on the evidence so far, more likely to be a good use of a reviewer's next hour than a page ranked near the bottom -- validated against a held-out set of clients the model never trained on.
- *Directional:* certain signals (search visibility, staleness, position, CTR gap) associate with decline/opportunity in this dataset; a model can rank on them better than a hand-written rule, measured by precision@50 against an observed label.

**What I will not claim, ever:**
- That a refresh *caused* a recovery -- this data has no experiment behind it, only observation. I'd need an A/B design or a before/after causal setup to say that.
- That I've found or predicted anything about Google's ranking algorithm, or about AI search citations/rankings -- I only have observable search and engagement signals, never the algorithm itself.
- That `is_declining_label` (a same-window bucket derived from `trend_direction`/`trend_pct`) is the true target for the capstone. It's a fine label for the Week-1 starter pipeline, but a stronger version of this lane needs a *future-window* label -- prior 90 days of features predicting the next 30 days -- so the model is forecasting, not just describing the window it already saw. That's a Week-3/4 data-contract problem, not a Week-1 one, but naming it now so it doesn't quietly become the capstone target by accident.




In [14]:
claim_guardrails = {
    "can_claim": [
        "observed patterns in the available metrics",
        "decision-support ranking for reviewers",
        "directional association between signals and review priority",
    ],
    "cannot_claim": [
        "a refresh caused a recovery",
        "anything about Google's ranking algorithm",
        "same-window labels as proof of future decline",
    ],
    "feature_rules": {
        "exclude_from_decline_model": ["trend_pct", "trend_direction"],
        "never_use_as_features": ["content_id", "client_id"],
    },
}

claim_guardrails


{'can_claim': ['observed patterns in the available metrics',
  'decision-support ranking for reviewers',
  'directional association between signals and review priority'],
 'cannot_claim': ['a refresh caused a recovery',
  "anything about Google's ranking algorithm",
  'same-window labels as proof of future decline'],
 'feature_rules': {'exclude_from_decline_model': ['trend_pct',
   'trend_direction'],
  'never_use_as_features': ['content_id', 'client_id']}}

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
